# DSCR (Debt Service Coverage Ratio) — Sample Data Calculation

Step-by-step walkthrough of the DSCR metric using the same sample data
the TypeScript calculation engine uses. This notebook serves as a Python
reference implementation for the metric.

**Formula:** `CASE(CRE: NOI, C&I: EBITDA) / Total Debt Service` — exposure-weighted at rollup levels

**Product-type branching:**
- CRE products (`BL`, `BRIDGE`) → use `noi_amt` as numerator
- C&I products (everything else) → use `ebitda_amt` as numerator

## 1. Setup — Load Sample Data

In [ ]:
import json
import os
import pandas as pd

# Resolve paths relative to this notebook
SCRIPTS_DIR = os.path.dirname(os.path.abspath('__file__'))
L1_PATH = os.path.join(SCRIPTS_DIR, 'l1', 'output', 'sample-data.json')
L2_PATH = os.path.join(SCRIPTS_DIR, 'l2', 'output', 'sample-data.json')

with open(L1_PATH) as f:
    l1_data = json.load(f)
with open(L2_PATH) as f:
    l2_data = json.load(f)

def to_df(data: dict, table_key: str) -> pd.DataFrame:
    """Convert {columns, rows} sample data entry to a DataFrame."""
    entry = data[table_key]
    return pd.DataFrame(entry['rows'], columns=entry['columns'])

print(f'L1 tables: {len(l1_data)}')
print(f'L2 tables: {len(l2_data)}')

## 2. Load Source Tables

DSCR uses 4 source tables + 1 taxonomy for rollup.

In [ ]:
# L2: Financial data — contains NOI, EBITDA, total debt service
ffs = to_df(l2_data, 'L2.facility_financial_snapshot')
print(f'facility_financial_snapshot: {len(ffs)} rows')
ffs[['facility_id', 'as_of_date', 'noi_amt', 'ebitda_amt', 'total_debt_service_amt']].head(10)

In [ ]:
# L2: Exposure data — provides weighting basis for rollup
fes = to_df(l2_data, 'L2.facility_exposure_snapshot')
print(f'facility_exposure_snapshot: {len(fes)} rows')
fes[['facility_id', 'as_of_date', 'gross_exposure_usd', 'drawn_amount', 'committed_amount']].head(10)

In [ ]:
# L1: Facility master — links facility to counterparty, product, and business segment
fm = to_df(l1_data, 'L1.facility_master')
print(f'facility_master: {len(fm)} rows')
fm[['facility_id', 'counterparty_id', 'product_node_id', 'lob_segment_id']].head(10)

In [ ]:
# L1: Product taxonomy — determines CRE vs C&I classification
ept = to_df(l1_data, 'L1.enterprise_product_taxonomy')
print(f'enterprise_product_taxonomy: {len(ept)} rows')
ept[['product_node_id', 'product_code', 'product_name', 'tree_level']]

## 3. Join Tables & Apply Product-Type Branching

Join path: `facility_exposure_snapshot` ← `facility_financial_snapshot` ← `facility_master` ← `enterprise_product_taxonomy`

Then apply: **CRE** (`BL`, `BRIDGE`) → `noi_amt` | **C&I** (others) → `ebitda_amt`

In [ ]:
AS_OF_DATE = '2025-01-31'

# Filter to as_of_date
fes_d = fes[fes['as_of_date'] == AS_OF_DATE].copy()
ffs_d = ffs[ffs['as_of_date'] == AS_OF_DATE].copy()

# Join: exposure ← financial (on facility_id + as_of_date)
merged = fes_d[['facility_id', 'as_of_date', 'gross_exposure_usd']].merge(
    ffs_d[['facility_id', 'as_of_date', 'noi_amt', 'ebitda_amt', 'total_debt_service_amt']],
    on=['facility_id', 'as_of_date'],
    how='left'
)

# Join: ← facility_master (for counterparty, product, segment)
merged = merged.merge(
    fm[['facility_id', 'counterparty_id', 'product_node_id', 'lob_segment_id']],
    on='facility_id',
    how='left'
)

# Join: ← product taxonomy (for product_code)
merged = merged.merge(
    ept[['product_node_id', 'product_code']],
    on='product_node_id',
    how='left'
)

# Apply CRE/C&I branching
CRE_CODES = ['BL', 'BRIDGE']
merged['is_cre'] = merged['product_code'].isin(CRE_CODES)
merged['cashflow'] = merged.apply(
    lambda r: r['noi_amt'] if r['is_cre'] else r['ebitda_amt'], axis=1
)

print(f'Joined rows: {len(merged)}')
merged[['facility_id', 'product_code', 'is_cre', 'noi_amt', 'ebitda_amt',
         'cashflow', 'total_debt_service_amt', 'gross_exposure_usd']].head(10)

## 4. Facility-Level DSCR

**Formula:** `cashflow / total_debt_service_amt`

At facility level, DSCR is a simple ratio — no weighting needed.

In [ ]:
import numpy as np

merged['dscr'] = np.where(
    merged['total_debt_service_amt'] > 0,
    merged['cashflow'] / merged['total_debt_service_amt'],
    0.0
)

facility_dscr = merged.groupby('facility_id').apply(
    lambda g: pd.Series({
        'product_code': g['product_code'].iloc[0],
        'is_cre': g['is_cre'].iloc[0],
        'numerator_used': 'noi_amt' if g['is_cre'].iloc[0] else 'ebitda_amt',
        'cashflow': g['cashflow'].iloc[0],
        'total_debt_service': g['total_debt_service_amt'].iloc[0],
        'gross_exposure': g['gross_exposure_usd'].iloc[0],
        'dscr': g['dscr'].iloc[0],
    })
).reset_index()

# This matches the TypeScript engine's exposure-weighted formula at facility level:
# SUM((cashflow / debt_service) * exposure) / SUM(exposure_where_debt_service > 0)
# At single-facility granularity this simplifies to cashflow / debt_service.

print('=== Facility-Level DSCR ===')
print(f'CRE facilities (use noi_amt): {facility_dscr["is_cre"].sum()}')
print(f'C&I facilities (use ebitda_amt): {(~facility_dscr["is_cre"].astype(bool)).sum()}')
print()
facility_dscr[['facility_id', 'product_code', 'numerator_used',
               'cashflow', 'total_debt_service', 'dscr']]

## 5. Counterparty Rollup — Exposure-Weighted Average

**Formula:** `SUM(facility_dscr × gross_exposure) / SUM(gross_exposure)` per counterparty

Only facilities with `total_debt_service > 0` contribute to the denominator.

In [ ]:
# Exposure-weighted DSCR rollup
has_debt_service = merged['total_debt_service_amt'] > 0

def weighted_dscr(group):
    mask = group['total_debt_service_amt'] > 0
    numerator = (group.loc[mask, 'dscr'] * group.loc[mask, 'gross_exposure_usd']).sum()
    denominator = group.loc[mask, 'gross_exposure_usd'].sum()
    return numerator / denominator if denominator > 0 else 0.0

cpty_dscr = merged.groupby('counterparty_id').apply(weighted_dscr).reset_index()
cpty_dscr.columns = ['counterparty_id', 'weighted_dscr']

print('=== Counterparty-Level Weighted DSCR ===')
cpty_dscr

## 6. Desk (L3) Rollup — via Business Taxonomy

Join `facility_master.lob_segment_id` → `enterprise_business_taxonomy.managed_segment_id`
to get the L3 desk name, then exposure-weighted average.

In [ ]:
# Load business taxonomy for desk names
ebt = to_df(l1_data, 'L1.enterprise_business_taxonomy')

# Join desk name
with_desk = merged.merge(
    ebt[['managed_segment_id', 'segment_name']],
    left_on='lob_segment_id',
    right_on='managed_segment_id',
    how='left'
)
with_desk['desk_name'] = with_desk['segment_name'].fillna('Unmapped Desk')

desk_dscr = with_desk.groupby('desk_name').apply(weighted_dscr).reset_index()
desk_dscr.columns = ['desk_name', 'weighted_dscr']
desk_dscr = desk_dscr.sort_values('desk_name')

print('=== Desk (L3) Level Weighted DSCR ===')
desk_dscr

## 7. Comparison with TypeScript Engine (API)

Compare Python results against the TypeScript calculation engine API.
Run the API call in a separate cell if the dev server is running on port 3000.

In [ ]:
import urllib.request

def call_dscr_api(dimension: str) -> dict:
    """Call the deep-dive API for DSCR at a given dimension."""
    url = 'http://localhost:3000/api/metrics/deep-dive/run'
    payload = json.dumps({'metricId': 'C101', 'dimension': dimension}).encode()
    req = urllib.request.Request(url, data=payload, headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=10) as resp:
            return json.loads(resp.read())
    except Exception as e:
        return {'ok': False, 'error': str(e)}

# Fetch API results
api_facility = call_dscr_api('facility')
api_l3 = call_dscr_api('L3')

if api_facility.get('ok'):
    api_fac_df = pd.DataFrame(api_facility['result']['rows'])
    comparison = facility_dscr[['facility_id', 'dscr']].merge(
        api_fac_df.rename(columns={'dimension_value': 'facility_id', 'metric_value': 'api_dscr'}),
        on='facility_id'
    )
    comparison['dscr'] = comparison['dscr'].astype(float).round(6)
    comparison['api_dscr'] = comparison['api_dscr'].astype(float).round(6)
    comparison['match'] = np.isclose(comparison['dscr'], comparison['api_dscr'], atol=1e-4)
    print('=== Facility-Level: Python vs API ===')
    display(comparison)
    print(f'\nAll match: {comparison["match"].all()}')
else:
    print(f'API not available: {api_facility.get("error", "unknown")}')
    print('Start dev server with: npm run dev')